In [28]:
from langgraph.graph import END, MessageGraph
from chains import responder_chain, revisor_chain
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, BaseMessage
from execute_tools import execute_tools
from typing import List

In [29]:
graph=MessageGraph()

C:\Users\user\AppData\Local\Temp\ipykernel_15780\1934083269.py:1: LangGraphDeprecatedSinceV10: MessageGraph is deprecated in LangGraph v1.0.0, to be removed in v2.0.0. Please use StateGraph with a `messages` key instead. Deprecated in LangGraph V1.0 to be removed in V2.0.
  graph=MessageGraph()


In [31]:
def responder(state):
    ans=responder_chain.invoke({"messages":state})
    return ans

def revisor(state):
    ans=revisor_chain.invoke({"messages":state})
    return ans

def execute_tool(state):
    ans=execute_tools(state)
    return ans
    

In [32]:
def should_continue(state: List[BaseMessage]) -> str:
    max_itr=2
    count=sum(isinstance(item,ToolMessage) for item in state)
    if count>=max_itr:
        return END
    else:
        return execute_tools

In [33]:
graph.add_node("draft",responder)
graph.add_node("revisor",revisor)
graph.add_node("execute_tools",execute_tools)
graph.set_entry_point("draft")
graph.add_edge("draft","execute_tools")
graph.add_edge("execute_tools","revisor")
graph.add_conditional_edges("revisor",should_continue)

In [34]:
app=graph.compile()

In [35]:
print(app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	draft(draft)
	revisor(revisor)
	execute_tools(execute_tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> draft;
	draft --> execute_tools;
	execute_tools --> revisor;
	revisor --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [36]:
result=app.invoke(HumanMessage(content="Write a 250 words blog post on how small eterprises can leverage AI to grow"))

Task revisor with path ('__pregel_pull', 'revisor') wrote to unknown channel branch:to:<function execute_tools at 0x0000015E0E345620>, ignoring it.


In [38]:
result

[HumanMessage(content='Write a 250 words blog post on how small eterprises can leverage AI to grow', additional_kwargs={}, response_metadata={}, id='ab3c0452-fa62-4f45-9a0a-efb782ae9636'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'a51ws8qes', 'function': {'arguments': '{"answer":"Small enterprises can leverage AI to grow in various ways. One of the primary ways is by automating repetitive and mundane tasks, such as data entry, bookkeeping, and customer service. AI-powered tools like chatbots and virtual assistants can help small businesses save time and resources, allowing them to focus on more strategic and creative tasks. Additionally, AI can help small enterprises analyze large amounts of data, providing valuable insights that can inform business decisions. For instance, AI-powered analytics tools can help businesses identify trends, predict customer behavior, and optimize marketing campaigns. Furthermore, AI can also help small enterprises improve their custo

In [37]:
print(result[-1].tool_calls[0]["args"]["answer"])

Small enterprises can leverage AI to grow by automating tasks, analyzing data, and improving customer experience. AI-powered tools like chatbots and virtual assistants can save time and resources. AI can also help small enterprises analyze data, providing valuable insights to inform business decisions. Additionally, AI can help small enterprises improve their customer experience by providing personalized recommendations and services. According to a study by McKinsey, AI can help businesses increase revenue by up to 20% (McKinsey, 2020). Another study by Harvard Business Review found that AI can help small businesses improve their operations and reduce costs (Harvard Business Review, 2019).
